# Abhay: local Windows training and validation

This notebook uses your downloaded competition features. It compares Dalvin's final unknown rule, detector-only scoring, and Abhay's learned fusion on identical candidates and trained stacking models. It uses the same identity-disjoint training/calibration/validation protocol as the Kaggle notebook.

**No GPU is required.** Keep the PC awake during training. SIFT matching and training can take significant CPU time; no real-data runtime has been benchmarked. Raw photos are unused.

Expected Downloads layout:
```
Downloads/
  reid_student_data_main/       # five core files
  sift_features_train.npz/      # extracted *_desc.npy and *_xy.npy arrays
  sift_features_test.npz/
```

Recommended first-time setup in PowerShell, from your `Learning/Models/Abhay` folder:
```
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt jupyterlab ipykernel
.\.venv\Scripts\python.exe -m jupyter lab
```
Open this notebook in JupyterLab. If using VS Code, select that `.venv` as the notebook kernel. Run the cells in order. The optional install switch below installs into the currently selected kernel environment.


In [3]:
import sys, subprocess, importlib.util
from importlib.metadata import version

INSTALL_PACKAGES = False  # Set True only if you want to install/update this kernel's dependencies.
requirements = ["numpy>=1.26,<2.5", "pandas>=2.1,<3", "scipy>=1.11,<2",
                "scikit-learn>=1.4,<2", "lightgbm>=4.3,<5",
                "opencv-python-headless>=4.8,<5", "joblib>=1.3,<2"]
if INSTALL_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])
    print("Dependencies installed. Restart the kernel, set INSTALL_PACKAGES=False, then run again.")
    raise RuntimeError("Restart the kernel after installing dependencies.")
modules = ["numpy", "pandas", "scipy", "sklearn", "lightgbm", "cv2", "joblib"]
missing = [name for name in modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Missing {missing}. Select the prepared .venv kernel or enable INSTALL_PACKAGES above.")
print("Python kernel:", sys.executable)
for package in ("numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "joblib"):
    print(package, version(package))


Python kernel: c:\Users\SANGEETA-DAILY\AppData\Local\Programs\Python\Python312\python.exe
numpy 2.4.6
pandas 2.3.0
scipy 1.15.3
scikit-learn 1.7.0
lightgbm 4.7.0
joblib 1.5.1


## 1. Create this run's folder and write the embedded code
Each execution creates a new timestamped run folder, preserving previous results. The source embedded here matches the Kaggle implementation. The dataset stays in Downloads and is read only.


In [4]:
from pathlib import Path
from datetime import datetime

ABHAY_FOLDER = Path(r"C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay")
RUN_DIR = ABHAY_FOLDER / "local_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
SCRIPT_DIR = RUN_DIR / "code"
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
(SCRIPT_DIR / 'dalvin_core.py').write_text('"""Adapted Dalvin Model 3b functions. See README.md for attribution and changes."""\nimport os\n\nimport re\n\nfrom multiprocessing import Pool\n\nimport cv2\n\nimport lightgbm as lgb\n\nimport numpy as np\n\nimport pandas as pd\n\nfrom scipy import sparse\n\nfrom scipy.sparse.csgraph import connected_components\n\nfrom sklearn.decomposition import PCA\n\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\n\n\n\n\ndef unit(a):\n    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)\n\ndef embed(gallery, labels, queries):\n    pca = PCA(min(384, gallery.shape[1], len(gallery) + len(queries) - 1), random_state=0).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)\n    project = lambda a: unit(lda.transform(pca.transform(a)))\n    return project(gallery), project(queries)\n\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s gallery images."""\n    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 8).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n\ndef build_frames(gi, qi):\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q), cs.append(c), order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))\n\ndef duplicate_groups(ix, link):\n    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef columns(s):\n    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\ndef fit(sims):\n    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": min(300, PARAMS["n_estimators"]), "num_leaves": 15}).fit(\n        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\ndef rank(s, r):\n    """Stable top-five; ties preserve the conditional known-identity ordering."""\n    nq = len(s["q"])\n    known = s["c"].ident.to_numpy(dtype=object).reshape(nq, K)\n    order = np.argsort(-r["pc"], axis=1, kind="stable")\n    labels = np.concatenate([np.take_along_axis(known, order, 1), np.full((nq, 1), "new_individual", dtype=object)], 1)\n    weights = np.take_along_axis((1-r["pb"])[:, None]*r["pc"], order, 1)\n    scores = np.concatenate([weights, r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, axis=1, kind="stable")[:, :5], 1)\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])\n\ndef augment(s, r):\n    """Add the previous round\'s predictions for each query and for its nearest query neighbours."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)\n    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours\n        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []\n        for k in range(5):\n            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale\n            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))\n        pb, w = np.stack(pb, 1), np.stack(w, 1)\n        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]\n        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()\n    return dict(s, q=q, c=c)\n\nK = 15\nNEIGHBOURS = 150\nSIFT_PULL = 0.05\nVIEWS = ["left", "right", "front", "back", "frontleft", "frontright", "backleft", "backright", "up", "down"]\nPARAMS = dict(objective="binary", learning_rate=.05, num_leaves=31, min_child_samples=30, subsample=.8, subsample_freq=1, colsample_bytree=.8, n_estimators=400, verbose=-1, random_state=42, n_jobs=2)\nINDEX_COLS = re.compile(r"^(nb|nw|sb|sw)\\d$")\nREGIMES = [None, (0.95, 100), (0.99, 80), (0.9, 60)]\n', encoding="utf-8")
(SCRIPT_DIR / 'train_eval.py').write_text('"""Abhay: frozen Dalvin retrieval/ranking + learned open-set score fusion.\n\nRun --help. Validation is identity-disjoint from every supervised training stage.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\n\nimport cv2\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nimport dalvin_core as b\n\nNEW = "new_individual"\n\n\ndef find_sift(root, split):\n    """Kaggle mounts SIFT beside the metadata folder, sometimes as extracted NPZ directories."""\n    name = f"sift_features_{split}.npz"\n    for parent in (root, root.parent, root.parent.parent):\n        candidate = parent / name\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Cannot find {name} in {root} or its two parent directories. "\n                            "Attach the full projectwild3 competition input, including both SIFT folders.")\n\n\nclass SiftSource:\n    def __init__(self, path):\n        self.archive = None\n        self.files = {}\n        if path.is_file():\n            self.archive = np.load(path)\n        else:\n            for file in path.rglob("*.npy"):\n                if file.stem in self.files:\n                    raise ValueError(f"Duplicate SIFT key {file.stem} in {path}")\n                self.files[file.stem] = file\n            if not self.files:\n                raise ValueError(f"No extracted .npy SIFT arrays in {path}")\n\n    def __getitem__(self, key):\n        return self.archive[key] if self.archive is not None else np.load(self.files[key], allow_pickle=False)\n\n\ndef load_data(root):\n    train = pd.read_csv(root / "train.csv")\n    test = pd.read_csv(root / "test.csv")\n    ids = pd.read_csv(root / "miewid_ids.csv")\n    all_df = pd.concat([train, test.assign(identity=None)], ignore_index=True)\n    if not all_df.id.is_unique or not ids.id.is_unique:\n        raise ValueError("Duplicate image IDs")\n    for col in ("species", "viewpoint", "identity"):\n        if col not in all_df:\n            raise ValueError(f"Missing column: {col}")\n    if train.identity.isna().any() or not set(all_df.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid identity/species metadata")\n    raw = np.load(root / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(ids) or set(ids.id) != set(all_df.id):\n        raise ValueError("Feature ID mapping does not match metadata")\n    ix = pd.Index(ids.id).get_indexer(all_df.id)\n    X = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(X).all() or np.any(np.linalg.norm(X, axis=1) == 0):\n        raise ValueError("Features contain nonfinite values or zero vectors")\n    b.X = b.unit(X)\n    b.IDS = all_df.id.to_numpy()\n    b.Y = all_df.identity.to_numpy(dtype=object)\n    b.SP = all_df.species.to_numpy()\n    b.VIEW = all_df.viewpoint.map({v: i for i, v in enumerate(b.VIEWS)}).fillna(-1).to_numpy(int)\n    return train, test, all_df\n\n\nclass SiftCache:\n    """Role-local neighbour selection; thread workers share lazily loaded descriptors."""\n    def __init__(self, root, out, workers, neighbours):\n        paths = [find_sift(root, s) for s in ("train", "test")]\n        print("SIFT sources:", *paths, sep="\\n", flush=True)\n        self.z = [SiftSource(path) for path in paths]\n        self.read_lock = Lock()\n        self.workers, self.neighbours = workers, neighbours\n        fingerprint = hashlib.sha256(b.IDS.tobytes() if b.IDS.dtype != object else "\\n".join(b.IDS).encode())\n        fingerprint.update(np.ascontiguousarray(b.X).tobytes())\n        self.path = out / ("sift_" + fingerprint.hexdigest()[:16] + ".npz")\n        self.done = {}\n        if self.path.exists():\n            with np.load(self.path) as z:\n                self.done = {tuple(p): tuple(v) for p, v in zip(z["pairs"], z["values"])}\n\n    @lru_cache(maxsize=1200)\n    def descriptors(self, i):\n        iid = b.IDS[i]\n        z = self.z[0 if iid.startswith("tr_") else 1]\n        # NpzFile\'s shared ZIP handle is not safe for concurrent reads.\n        with self.read_lock:\n            d = np.asarray(z[f"{iid}_desc"], dtype=np.float32)\n            xy = np.asarray(z[f"{iid}_xy"], dtype=np.float32)\n        if d.ndim != 2 or xy.shape != (len(d), 2):\n            raise ValueError(f"Malformed SIFT arrays for {iid}")\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1))\n        return d, xy\n\n    def verify(self, pair):\n        a, c = pair\n        da, xa = self.descriptors(a)\n        dc, xc = self.descriptors(c)\n        if min(len(da), len(dc)) < 4:\n            return 0, 0\n        matches = cv2.BFMatcher().knnMatch(da, dc, k=2)\n        good = [m[0] for m in matches if len(m) == 2 and m[0].distance < .8*m[1].distance]\n        if len(good) < 4:\n            return len(good), 0\n        # Local RNG seed per task. OpenCV RNG state is thread-local.\n        cv2.setRNGSeed(int((int(a)*100003 + int(c)) % (2**31-1)))\n        _, mask = cv2.estimateAffinePartial2D(\n            np.float32([xa[m.queryIdx] for m in good]),\n            np.float32([xc[m.trainIdx] for m in good]),\n            method=cv2.RANSAC, ransacReprojThreshold=10.)\n        return len(good), int(mask.sum()) if mask is not None else 0\n\n    def activate(self, indices):\n        wanted = set()\n        for sp in ("hyena", "leopard"):\n            ix = indices[b.SP[indices] == sp]\n            # Chunked similarities avoid allocating the full role x role matrix.\n            for start in range(0, len(ix), 256):\n                q = ix[start:start+256]\n                s = b.X[q] @ b.X[ix].T\n                s[np.arange(len(q)), np.arange(start, start+len(q))] = -np.inf\n                k = min(self.neighbours, len(ix)-1)\n                nb = np.argsort(-s, axis=1)[:, :k]\n                for a, row in zip(q, nb):\n                    wanted.update((min(int(a), int(c)), max(int(a), int(c))) for c in ix[row])\n        missing = sorted(wanted - self.done.keys())\n        print(f"SIFT: {len(wanted)} role-local pairs; {len(missing)} uncached", flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j, (pair, value) in enumerate(zip(missing, pool.map(self.verify, missing)), 1):\n                self.done[pair] = value\n                if j % 10000 == 0:\n                    print(f"  verified {j}/{len(missing)}", flush=True)\n        pairs = sorted(self.done)\n        tmp = self.path.with_suffix(".tmp.npz")\n        np.savez_compressed(tmp, pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n                            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n        os.replace(tmp, self.path)\n        # Only expose requested pairs, even when cache contains other roles/test.\n        pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n        vv = np.array([self.done[tuple(p)] for p in pp]).reshape(-1, 2)\n        n = len(b.X)\n        def matrix(v):\n            a, c = pp.T\n            return sparse.coo_matrix((np.r_[v, v], (np.r_[a, c], np.r_[c, a])), shape=(n, n)).tocsr()\n        b.GOOD, b.INL = matrix(vv[:, 0]), matrix(vv[:, 1])\n\n\ndef allocate_roles(train, rounds, seed):\n    rng = np.random.default_rng(seed)\n    roles = {f"stage{i+1}": [] for i in range(rounds)} | {"calibration": [], "validation": []}\n    for sp in ("hyena", "leopard"):\n        names = np.array(sorted(train.loc[train.species == sp, "identity"].unique()))\n        rng.shuffle(names)\n        n = max(1, int(round(.2*len(names))))\n        roles["validation"].extend(names[:n])\n        roles["calibration"].extend(names[n:2*n])\n        for i, block in enumerate(np.array_split(names[2*n:], rounds)):\n            roles[f"stage{i+1}"].extend(block)\n    sets = [set(v) for v in roles.values()]\n    assert all(not a & c for i, a in enumerate(sets) for c in sets[i+1:])\n    return {k: np.flatnonzero(train.identity.isin(v).to_numpy()) for k, v in roles.items()}\n\n\ndef split_episode(indices, seed, cap=32, link=(.95, 100)):\n    """Unknown identities absent from gallery; known-query duplicate groups kept whole."""\n    rng = np.random.default_rng(seed)\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        ix = indices[b.SP[indices] == sp]\n        names = np.unique(b.Y[ix])\n        rng.shuffle(names)\n        nu = max(1, int(round(rng.uniform(.2, .35)*len(names))))\n        unseen = set(names[:nu])\n        for name in names:\n            rows = ix[b.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in unseen:\n                unknown.extend(rows[:cap])\n                continue\n            held = []\n            groups = b.duplicate_groups(rows, link)\n            target = min(max(1, int(round(.35*len(rows)))), cap)\n            for c in rng.permutation(np.unique(groups)):\n                members = rows[groups == c].tolist()\n                if len(held) < target and len(held)+len(members) < len(rows):\n                    held.extend(members)\n            # Never truncate a duplicate group after selecting it.\n            known.extend(held)\n            gallery.extend(i for i in rows if i not in set(held))\n    gi, qi = np.array(gallery, dtype=int), np.array(known+unknown, dtype=int)\n    if not len(known) or not len(unknown):\n        raise ValueError("Episode lacks known/unknown queries; use a larger role or different seed")\n    for sp in ("hyena", "leopard"):\n        if len(np.unique(b.Y[gi[b.SP[gi] == sp]])) < b.K or np.sum(b.SP[qi] == sp) < 6:\n            raise ValueError("Too few identities/queries per species; reduce --rounds or --candidates")\n    assert not set(b.Y[unknown]) & set(b.Y[gi])\n    assert set(b.Y[known]) <= set(b.Y[gi])\n    return gi, qi, np.where(np.isin(qi, unknown), NEW, b.Y[qi])\n\n\ndef frame(gi, qi, truth=None):\n    s = b.build_frames(gi, qi)\n    if truth is not None:\n        lookup = dict(zip(qi, truth))\n        s["truth"] = np.array([lookup[i] for i in s["order"]])\n        s["q"]["unk"] = (s["truth"] == NEW).astype(int)\n        s["c"]["y"] = (s["c"].ident.to_numpy() == s["truth"][s["c"].q.to_numpy()]).astype(int)\n    return s\n\n\ndef through_stack(s, models):\n    for i, m in enumerate(models):\n        r = b.predict(m, s)\n        if i < len(models)-1:\n            s = b.augment(s, r)\n    return s, r\n\n\ndef fusion_features(s, r):\n    """Only inference-available evidence; no identity strings, labels or image indices."""\n    q = s["q"]\n    pc = r["pc"]\n    top = pc.argmax(1)\n    rows = np.arange(len(q))\n    c = s["c"].iloc[rows*b.K+top]\n    ordered = -np.sort(-pc, axis=1)\n    return pd.DataFrame({\n        "detector": r["pn"], "legacy_new": r["pb"],\n        "raw_sum": r["raw"].sum(1), "raw_max": r["raw"].max(1),\n        "pc_max": ordered[:, 0], "pc_gap": ordered[:, 0]-ordered[:, 1],\n        "pc_entropy": -(pc*np.log(np.maximum(pc, 1e-12))).sum(1),\n        "raw_similarity": c.mxr.to_numpy(), "lda_similarity": c.mxl.to_numpy(),\n        "inliers": np.log1p(c.sx.to_numpy()), "sift_support": np.log1p(c.support.to_numpy()),\n        "same_view_similarity": c.same.to_numpy(), "same_view_available": c.same.notna().to_numpy().astype(int),\n        "gallery_support": np.log1p(c.cnt.to_numpy()), "hyena": q.sp.to_numpy(),\n        "raw_margin": q.rm12.to_numpy(), "lda_margin": q.lm12.to_numpy(),\n    })\n\n\ndef probabilities(s, r, fusion):\n    return {"dalvin_rule": r["pb"], "detector_only": r["pn"],\n            "abhay_fusion": fusion.predict_proba(fusion_features(s, r))[:, 1]}\n\n\ndef evaluate(s, r, probs, all_df, output):\n    meta = all_df.iloc[s["order"]].reset_index(drop=True)\n    unknown = s["truth"] == NEW\n    result = pd.DataFrame({"id": meta.id, "identity": meta.identity, "species": meta.species,\n                           "viewpoint": meta.viewpoint, "truth": s["truth"], "unknown": unknown})\n    labels = s["c"].ident.to_numpy().reshape(-1, b.K)\n    result["candidate_hit"] = (labels == s["truth"][:, None]).any(1)\n    reports = {}\n    for name, p in probs.items():\n        preds = b.rank(s, dict(r, pb=p))\n        score = np.array([next((1/(j+1) for j, v in enumerate(row) if v == t), 0.) for row, t in zip(preds, s["truth"])])\n        result[f"{name}_p_new"] = p\n        result[f"{name}_ap5"] = score\n        result[f"{name}_predictions"] = [" ".join(row) for row in preds]\n        reports[name] = {"map5": float(score.mean()), "known_map5": float(score[~unknown].mean()),\n                         "unknown_map5": float(score[unknown].mean()),\n                         "known_new_at1": float(np.mean(preds[~unknown, 0] == NEW)),\n                         "unknown_new_at1": float(np.mean(preds[unknown, 0] == NEW))}\n    result.to_csv(output / "validation_predictions.csv", index=False)\n    slices = []\n    for fields in (["species", "unknown"], ["viewpoint", "unknown"]):\n        for key, group in result.groupby(fields, dropna=False):\n            slices.append({"slice": str(dict(zip(fields, key))), "n": len(group),\n                           **{name: float(group[f"{name}_ap5"].mean()) for name in probs}})\n    pd.DataFrame(slices).to_csv(output / "validation_slices.csv", index=False)\n    return {"n_queries": len(result), "unknown_fraction": float(unknown.mean()),\n            "known_candidate_recall": float(result.loc[~unknown, "candidate_hit"].mean()),\n            "methods": reports, "fusion_delta_map5": reports["abhay_fusion"]["map5"]-reports["dalvin_rule"]["map5"]}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument("--data", type=Path, required=True)\n    ap.add_argument("--output", type=Path, default=Path("abhay_output"))\n    ap.add_argument("--seed", type=int, default=2026)\n    ap.add_argument("--rounds", type=int, choices=[1, 2, 3], default=3)\n    ap.add_argument("--sims", type=int, default=12, help="simulations per identity stage")\n    ap.add_argument("--calibration-sims", type=int, default=4)\n    ap.add_argument("--neighbours", type=int, default=150)\n    ap.add_argument("--candidates", type=int, default=15)\n    ap.add_argument("--trees", type=int, default=400)\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--predict-test", action="store_true", help="after validation, also write all three submissions")\n    args = ap.parse_args()\n    if min(args.sims, args.calibration_sims, args.neighbours, args.workers, args.trees) < 1 or args.candidates < 5:\n        ap.error("counts must be positive and --candidates >= 5")\n    args.output.mkdir(parents=True, exist_ok=True)\n    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\n    config["python"] = platform.python_version()\n    config["packages"] = {n: version(n) for n in ("numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "joblib")}\n    (args.output / "config.json").write_text(json.dumps(config, indent=2))\n    np.random.seed(args.seed)\n    cv2.setNumThreads(1)\n    b.K = args.candidates\n    b.PARAMS.update(n_estimators=args.trees, n_jobs=args.workers, random_state=args.seed)\n    train, test, all_df = load_data(args.data)\n    roles = allocate_roles(train, args.rounds, args.seed)\n    manifest = []\n    for name, ix in roles.items():\n        manifest.extend({"id": b.IDS[i], "identity": b.Y[i], "species": b.SP[i], "role": name} for i in ix)\n    pd.DataFrame(manifest).to_csv(args.output / "identity_roles.csv", index=False)\n    cache = SiftCache(args.data, args.output, args.workers, args.neighbours)\n    models = []\n    split_rows = []\n    def episode(ix, seed, role, link):\n        gi, qi, truth = split_episode(ix, seed, link=link)\n        truth_map = dict(zip(qi, truth))\n        gallery_set = set(gi)\n        split_rows.extend({"episode": seed, "role": role, "id": b.IDS[i],\n                           "assignment": "gallery" if i in gallery_set else "unknown_query" if truth_map[i] == NEW else "known_query"}\n                          for i in np.r_[gi, qi])\n        return frame(gi, qi, truth)\n    for stage in range(args.rounds):\n        name = f"stage{stage+1}"\n        ix = roles[name]\n        cache.activate(ix)\n        sims = []\n        for j in range(args.sims):\n            s = episode(ix, args.seed+1000*(stage+1)+j, name, b.REGIMES[j % len(b.REGIMES)])\n            for model in models:\n                r = b.predict(model, s)\n                s = b.augment(s, r)\n            sims.append(s)\n        print(f"Fitting {name}: {len(ix)} images, {len(sims)} simulations", flush=True)\n        models.append(b.fit(sims))\n    cache.activate(roles["calibration"])\n    cx, cy, cw = [], [], []\n    for j in range(args.calibration_sims):\n        s = episode(roles["calibration"], args.seed+10000+j, "calibration", (.95, 100))\n        s, r = through_stack(s, models)\n        cx.append(fusion_features(s, r))\n        cy.append(s["truth"] == NEW)\n        cw.extend(b.IDS[s["order"]])\n    Xc, yc = pd.concat(cx, ignore_index=True), np.concatenate(cy).astype(int)\n    # An image can recur in calibration episodes: total weight per image is one.\n    counts = pd.Series(cw).value_counts()\n    weights = np.array([1/counts[i] for i in cw])\n    fusion = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),\n                           LogisticRegression(C=1., max_iter=3000, random_state=args.seed))\n    fusion.fit(Xc, yc, logisticregression__sample_weight=weights)\n    feature_names = fusion[0].get_feature_names_out(Xc.columns)\n    pd.DataFrame({"feature": feature_names, "standardized_coefficient": fusion[-1].coef_[0]}).to_csv(\n        args.output / "fusion_coefficients.csv", index=False)\n    pd.DataFrame({"id": cw, "unknown": yc, "sample_weight": weights}).to_csv(args.output / "calibration_examples.csv", index=False)\n    # Save frozen models before final validation. Validation cannot affect fitting.\n    joblib.dump({"models": models, "fusion": fusion, "config": config, "fusion_columns": list(Xc)}, args.output / "models.joblib")\n    cache.activate(roles["validation"])\n    s = episode(roles["validation"], args.seed+20000, "validation", (.95, 100))\n    s, r = through_stack(s, models)\n    report = evaluate(s, r, probabilities(s, r, fusion), all_df, args.output)\n    report["config"] = config\n    report["protocol"] = "identity-disjoint sequential stacking stages, calibration, outer validation"\n    (args.output / "metrics.json").write_text(json.dumps(report, indent=2))\n    pd.DataFrame(split_rows).to_csv(args.output / "episode_splits.csv", index=False)\n    print(json.dumps(report, indent=2), flush=True)\n    if args.predict_test:\n        # Evaluation is complete; only now expose competition test queries.\n        cache.activate(np.arange(len(all_df)))\n        s, r = through_stack(frame(np.arange(len(train)), np.arange(len(train), len(all_df))), models)\n        sample = pd.read_csv(args.data / "sample_submission.csv")\n        for name, p in probabilities(s, r, fusion).items():\n            preds = b.rank(s, dict(r, pb=p))\n            sub = pd.DataFrame({"id": b.IDS[s["order"]], "predictions": [" ".join(v) for v in preds]})\n            sub = sample[["id"]].merge(sub, on="id", how="left", validate="one_to_one")\n            if len(sub) != len(test) or set(sub.id) != set(test.id) or sub.predictions.isna().any():\n                raise ValueError("Submission IDs missing or invalid")\n            known = train.groupby("species").identity.apply(set).to_dict()\n            species = test.set_index("id").species\n            for row in sub.itertuples():\n                tokens = row.predictions.split()\n                assert len(tokens) == len(set(tokens)) == 5\n                assert all(t == NEW or t in known[species[row.id]] for t in tokens)\n            sub.to_csv(args.output / f"submission_{name}.csv", index=False)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(SCRIPT_DIR / 'README.md').write_text("# Abhay: learned open-set fusion on Dalvin Model 3b\n\nThis experiment replaces the **final** unknown-animal scoring rule with logistic regression. It compares the legacy rule, the existing unknown detector alone, and learned fusion on the same candidates, trained stack, queries, and known-identity scores. No improvement is claimed until the competition-data experiment finishes.\n\n## Attribution and scope\n\nDalvin wrote `Models/Dalvin/reid_pipeline.py` (Model 3b). `dalvin_core.py` contains its retrieval, feature engineering, LightGBM models, ranking, duplicate grouping, and stacking functions. `provenance.json` records the source hash. Abhay's addition is `train_eval.py`, the calibration comparison, independent validation protocol, Kaggle notebook, diagnostics, and tests.\n\nRetained: normalized MIEW-ID; PCA then shrinkage LDA per species; raw/projected/centroid similarities; RootSIFT ratio test and partial-affine RANSAC; 15 candidate identities; three LightGBM classifiers; three stacking rounds. Earlier stacking rounds continue to use Dalvin's legacy probabilities in every comparison. Only the final probability of `new_individual` differs, preserving the relative order of known identities.\n\nOriginal rule: `p_new = (detector_probability + clip(1 - sum(binary_candidate_scores), 0, 1)) / 2`. Candidate binary scores need not sum to a meaningful known-animal probability. Fusion learns from detector output, legacy score, candidate scores and uncertainty, raw/LDA margins, SIFT support, viewpoint availability, gallery support, and species. It uses median imputation, scaling, and fixed-C logistic regression. This is learned score fusion; calibrated probabilities and better MAP@5 are hypotheses to test, not guarantees.\n\n## Validation design: read before running\n\nThe original simulation-level folds can reuse the same underlying identities across training/evaluation. This runner deliberately changes that training protocol for **all three comparison methods**:\n\n1. Shuffle identities independently within each species, seed 2026.\n2. Allocate 20% to calibration, 20% to outer validation, and 60% to stack training. Divide the training identities into three disjoint stage blocks.\n3. Train round 1 on simulated episodes from stage 1. Generate round-1 predictions on stage 2, augment them, then train round 2 there. Generate round-1 and round-2 predictions on stage 3, then train round 3 there. Earlier models have never trained on later-stage identities.\n4. Run the frozen three-round stack on calibration episodes. Fit fusion there, without refitting any base model. Repeated sightings across calibration episodes receive inverse occurrence weights, giving each sighting total weight one.\n5. Save the frozen models. Run one separate outer validation episode. Neither its identities nor labels entered any supervised model or fusion fit. All three methods are evaluated once on the same queries. There is no validation-based automatic method selection or hyperparameter search.\n\nWithin each episode, 20-35% of identities are absent from the gallery: their query truth is `new_individual`. Remaining identities have catalogue images and, where possible, held-out known queries. Entire near-duplicate connected components are held out together; selected groups are never truncated. Groups use provided raw cosine and cached SIFT evidence; unverified pairs mean this is only an approximate duplicate detector, not a guarantee of encounter separation. Singletons and identities with one connected component can remain gallery-only. Query caps affect the mix; actual counts and unknown fraction are reported.\n\nThe final validation episode uses raw cosine >0.95 or >=100 SIFT inliers for grouping. Training mixes Dalvin's four grouping regimes. Calibration uses the stricter fixed grouping regime. Identity split labels define the experiment only; they are never predictor features. PCA may see that episode's unlabelled queries, and LDA sees that episode's gallery labels, as in Dalvin's transductive approach. Gallery-label use is part of retrieval, not fusion/ranker training on validation truth. No competition test query participates in validation retrieval, neighbour stacking, or PCA.\n\nThis sequential identity holdout uses fewer identities per stage and smaller galleries than the original full-data run. Its `dalvin_rule` result is a **controlled baseline under this protocol**, not a reproduction of the team's 0.85078 public result or 0.906 local result. It trades training data efficiency for an understandable independent comparison. Gallery-size/prevalence shift remains a limitation. Keeping all three methods on the same trained stack isolates the fusion change.\n\nDo not tune repeatedly on `validation_predictions.csv` and still call it untouched validation. Use calibration/inner development episodes for changes, lock the method, and reserve a new outer evaluation for the final comparison. Do not assume the simulated unknown fraction equals the hidden test fraction.\n\n## Run on Kaggle\n\nUpload **`Abhay_OpenSet_Calibration.ipynb`** as a notebook. It embeds both Python modules, so you do not need to upload the repository as another dataset. Attach the competition dataset. CPU execution is sufficient; these are classical models, so a GPU does not accelerate the main work. Leave training running: SIFT matching and simulations can be expensive. No runtime estimate has been measured on the real data.\n\nAttach the existing **projectwild3 / ML Project 3** competition directly as a notebook input. A separate private dataset upload is unnecessary. Competition page: https://www.kaggle.com/competitions/projectwild3/data\n\nThe actual Kaggle layout has five core files under `reid_student_data_main` and SIFT sources beside that folder:\n\n```\n<competition input>/\n  reid_student_data_main/\n    train.csv\n    test.csv\n    miewid_ids.csv\n    miewid_features.npy\n    sample_submission.csv\n  sift_features_train.npz/   # May be an extracted folder containing *_desc.npy, *_xy.npy\n  sift_features_test.npz/\n```\n\nThe runner supports both original NPZ files and extracted directories named `sift_features_train.npz` and `sift_features_test.npz` (including nested NPY arrays). SIFT sources can be inside the core data directory or at either of its two parent levels. The notebook auto-detects the five-file metadata directory; it does not require all seven sources to share a folder. No moving or repacking of read-only competition files is needed. Original image directories are unused.\n\nIn the notebook input panel, attach the competition if it is not already listed. Seeing the files on the competition's Data page alone does not establish that a particular notebook has mounted that input. The notebook configuration cell lists mounted inputs and gives an explicit error if the core files cannot be found. Set `DATA` to the detected metadata directory if multiple copies are attached. Libraries are usually available on Kaggle; the optional install cell is only for missing dependencies.\n\nRun cells in order:\n\n1. Read the protocol, then write the embedded modules to the working directory.\n2. Set paths and configuration. Default: 3 rounds, 12 simulations per stage, 4 calibration episodes, 150 SIFT neighbours, 15 candidates, 400 trees (unknown detector capped at 300).\n3. Optionally run a fast **real-data wiring check** in a different output directory. Fewer trees/simulations/neighbours are for debugging and do not constitute a quality result.\n4. Run full training and evaluation. The command prints progress and final metrics.\n5. Inspect the summary, per-species/known-unknown/viewpoint slices, and error rows. Negative delta is a valid result; do not claim an improvement if it loses.\n6. Download the whole output directory with the notebook version. Record the parameters, source hash, seed, Kaggle version, and scores in the team log under your own commits.\n\nLocal equivalent, from this folder:\n\n```bash\npython -m pip install -r requirements.txt\npython train_eval.py --data /path/to/reid_student_data_main --output ./abhay_output\npython test_pipeline.py -v\n```\n\n`--sims 32` is a larger training experiment, not required to reproduce the default experiment. Keep other configuration fixed when comparing. If an identity block is too small, the runner fails with a clear error rather than silently changing candidate counts. `--rounds 1` is a simpler distinct experiment; do not mix its scores with the three-round run.\n\n## Outputs and interpretation\n\n| File | Purpose |\n|---|---|\n| `metrics.json` | Three methods' MAP@5, known/unknown MAP@5, rank-one unknown rates, candidate recall, delta, parameters and library versions |\n| `validation_predictions.csv` | Per-query truth, identity/species/viewpoint, candidate hit, p(new), predictions and reciprocal-rank score for every method |\n| `validation_slices.csv` | Scores and sample sizes by species/known-unknown and viewpoint/known-unknown |\n| `fusion_coefficients.csv` | Coefficients on standardized fusion inputs; correlated inputs limit causal interpretation |\n| `identity_roles.csv` | Every training image's exclusive supervised role |\n| `episode_splits.csv` | Gallery/known-query/unknown-query assignments for every episode |\n| `calibration_examples.csv` | Calibration queries, target and repeated-query weights |\n| `config.json`, `models.joblib` | Configuration and frozen trained stack/fusion; only load your own trusted checkpoint |\n| `sift_*.npz` | Reusable pair cache keyed by IDs and normalized global features; only role-local pairs are exposed in each phase |\n\n`fusion_delta_map5` is fusion MAP@5 minus legacy MAP@5 on the same validation queries. `known_new_at1` is the fraction of known queries incorrectly assigned new at rank one; smaller is better. `unknown_new_at1` is unknown detection at rank one; larger is better. MAP@5 is mean reciprocal rank through position five, not classification accuracy. Candidate recall is computed only for known queries. Small slice counts are unstable, and a single split cannot establish statistical superiority. Report counts and repeat a predeclared protocol on additional seeds before claiming generalization.\n\n## Optional test inference\n\nAdd `--predict-test` (or set `PREDICT_TEST=True` before the notebook run). After completing validation, the same frozen models predict test sightings using the full labelled gallery. The command writes `submission_dalvin_rule.csv`, `submission_detector_only.csv`, and `submission_abhay_fusion.csv`. It validates IDs, five distinct labels, allowed identities/species, and sample order. It does not upload to Kaggle. All three are saved so you can make a considered selection; none is automatically declared best.\n\nWeights are not refitted on calibration/validation after evaluation. The full final gallery is larger than the experimental galleries, so test performance can differ. Model selection and submission decisions belong to the team. The competition-test SIFT cache activates only after outer evaluation finishes.\n\n## Engineering differences from the source\n\n- No import-time data reads or implicit project paths. CLI paths work with Kaggle's input/output layout.\n- Features are aligned through `miewid_ids.csv`, not assumed to match concatenated CSV order.\n- PCA dimension is bounded by input dimension and seeded; zero-norm division is guarded.\n- LightGBM seed and thread limits are explicit. Default tree counts match the source; debug counts are configurable.\n- SIFT uses a portable thread pool with bounded lazy descriptor caching, role-local neighbour selection, chunked similarities and atomic cache writes. This avoids Windows multiprocessing globals and loading all SIFT arrays at once. Fixed per-pair RANSAC seeds improve repeatability; cross-platform numerical differences remain possible.\n- Final score ties use stable sorting with conditional known-identity order as the tie-break, shared by all comparison methods. This prevents changing p(new) from arbitrarily shuffling tied identities.\n- Source feature engineering still treats unverified SIFT pairs as zero and uses the original shortlist. These are intentionally separate future experiments.\n\n## Verification status\n\nSynthetic tests exercise shuffled ID alignment, empty test SIFT arrays, three actual LightGBM stacking rounds, learned fusion, disjoint roles and unknown/gallery identities, shared known-identity ordering, metrics and all three valid test CSVs. These tests verify execution and invariants, not competition performance. The real feature arrays were absent from the local clone, so a full real-data run and any performance claim remain pending your Kaggle run.\n", encoding="utf-8")
(SCRIPT_DIR / 'provenance.json').write_text('{\n  "source": "Models/Dalvin/reid_pipeline.py",\n  "source_sha256": "7e4f6be8981ef03955610014cd377d78ccdd3bff3e5d89864a4e466f312d0806",\n  "author": "Dalvin; adapted for Abhay calibration experiment",\n  "changes": [\n    "Removed import-time data loading and original main loop",\n    "Bound PCA components by feature dimension; fixed PCA seed",\n    "Guarded zero-vector normalization",\n    "Explicit deterministic LightGBM parameters; configurable threads and trees via runner",\n    "Original prediction, ranking, feature and stacking functions retained",\n    "Stable final ranking ties preserve conditional identity order for every comparison method"\n  ]\n}', encoding="utf-8")
(SCRIPT_DIR / 'requirements.txt').write_text('numpy>=1.26,<2.5\npandas>=2.1,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\nlightgbm>=4.3,<5\nopencv-python-headless>=4.8,<5\njoblib>=1.3,<2\n', encoding="utf-8")
print("Run folder:", RUN_DIR)


Run folder: C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay\local_runs\20261124_193658_932524


## 2. Check your Downloads dataset and configure training
Defaults match the Kaggle experiment: three stacking stages, 12 simulations per stage, four calibration episodes, 150 SIFT neighbours and 15 candidates. Two CPU workers are used. Larger settings can take substantial time and memory.

The preflight checks metadata counts, ID alignment, all required SIFT filenames and a sample descriptor from each split. It does not train models. If the files were downloaded to another location, edit DOWNLOADS. The optional small execution check writes separate results; its scores are not a quality benchmark.


In [5]:
import os, json
import numpy as np
import pandas as pd

DOWNLOADS = Path(r"C:\Users\SANGEETA-DAILY\Downloads")
DATA = DOWNLOADS / "reid_student_data_main"
required = ["train.csv", "test.csv", "miewid_ids.csv", "miewid_features.npy", "sample_submission.csv"]
missing = [name for name in required if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing core files in {DATA}: {missing}")
sys.path.insert(0, str(SCRIPT_DIR))
import train_eval as experiment
train_meta, test_meta, all_meta = experiment.load_data(DATA)
for split, metadata in (("train", train_meta), ("test", test_meta)):
    path = experiment.find_sift(DATA, split)
    source = experiment.SiftSource(path)
    keys = set(source.archive.files) if source.archive is not None else set(source.files)
    absent = [f"{iid}_{kind}" for iid in metadata.id for kind in ("desc", "xy")
              if f"{iid}_{kind}" not in keys]
    if absent:
        raise FileNotFoundError(f"{len(absent)} SIFT arrays missing from {path}; examples: {absent[:5]}")
    iid = metadata.id.iloc[0]
    print(split, "SIFT:", path, "sample descriptor shape:", source[f"{iid}_desc"].shape)
    if source.archive is not None:
        source.archive.close()
print("Images:", len(train_meta), "train,", len(test_meta), "test")
print("Feature shape:", experiment.b.X.shape)
OUTPUT = RUN_DIR / "full"
SEED, ROUNDS, SIMS, CALIBRATION_SIMS = 2026, 3, 12, 4
NEIGHBOURS, CANDIDATES, TREES = 150, 15, 400
WORKERS = min(2, os.cpu_count() or 1)
PREDICT_TEST = False
print("Results:", OUTPUT)


train SIFT: C:\Users\SANGEETA-DAILY\Downloads\sift_features_train.npz sample descriptor shape: (500, 128)
test SIFT: C:\Users\SANGEETA-DAILY\Downloads\sift_features_test.npz sample descriptor shape: (500, 128)
Images: 7163 train, 2791 test
Feature shape: (9954, 2152)
Results: C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay\local_runs\20261124_193658_932524\full


## 3. Optional fast wiring check
This still uses real competition features, but reduces the expensive settings. It verifies execution, not model quality. It writes to a separate folder. Leave disabled to proceed directly to the full experiment.


In [6]:
RUN_SMOKE = False  # Set True to check execution with reduced settings first.
if RUN_SMOKE:
    subprocess.run([sys.executable, str(SCRIPT_DIR / "train_eval.py"),
                    "--data", str(DATA), "--output", str(RUN_DIR / "smoke"),
                    "--sims", "1", "--calibration-sims", "1", "--neighbours", "10",
                    "--trees", "20", "--workers", str(WORKERS)], check=True)


## 4. Train, calibrate and evaluate
This launches a separate Python process using the same interpreter as your notebook kernel. Progress streams below. Keep the PC awake and this process running. No Kaggle access is required once dependencies and the dataset are downloaded.

The full model starts from scratch; SIFT pair caches inside the chosen output folder can be reused on rerun, but model-stage checkpoints are not used for resuming interrupted training. No estimated runtime is promised.


In [7]:
command = [sys.executable, str(SCRIPT_DIR / "train_eval.py"),
           "--data", str(DATA), "--output", str(OUTPUT), "--seed", str(SEED),
           "--rounds", str(ROUNDS), "--sims", str(SIMS),
           "--calibration-sims", str(CALIBRATION_SIMS), "--neighbours", str(NEIGHBOURS),
           "--candidates", str(CANDIDATES), "--trees", str(TREES), "--workers", str(WORKERS)]
if PREDICT_TEST:
    command.append("--predict-test")
print(" ".join(command))
subprocess.run(command, check=True)


c:\Users\SANGEETA-DAILY\AppData\Local\Programs\Python\Python312\python.exe C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay\local_runs\20261124_193658_932524\code\train_eval.py --data C:\Users\SANGEETA-DAILY\Downloads\reid_student_data_main --output C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay\local_runs\20261124_193658_932524\full --seed 2026 --rounds 3 --sims 12 --calibration-sims 4 --neighbours 150 --candidates 15 --trees 400 --workers 2


CompletedProcess(args=['c:\\Users\\SANGEETA-DAILY\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'C:\\Users\\SANGEETA-DAILY\\Documents\\Abhay\\Semester 5\\ML\\Project\\Learning\\Models\\Abhay\\local_runs\\20261124_193658_932524\\code\\train_eval.py', '--data', 'C:\\Users\\SANGEETA-DAILY\\Downloads\\reid_student_data_main', '--output', 'C:\\Users\\SANGEETA-DAILY\\Documents\\Abhay\\Semester 5\\ML\\Project\\Learning\\Models\\Abhay\\local_runs\\20261124_193658_932524\\full', '--seed', '2026', '--rounds', '3', '--sims', '12', '--calibration-sims', '4', '--neighbours', '150', '--candidates', '15', '--trees', '400', '--workers', '2'], returncode=0)

## 5. Compare results and inspect errors
A positive delta favours fusion on this split. Report both known and unknown results, counts, and the actual unknown fraction. A negative delta is a valid experiment outcome. Do not tune repeatedly against this outer validation set.


In [8]:
report = json.loads((OUTPUT / "metrics.json").read_text())
display(pd.DataFrame(report["methods"]).T)
print("Fusion minus baseline MAP@5:", report["fusion_delta_map5"])
print("Queries:", report["n_queries"], "Unknown fraction:", report["unknown_fraction"])
print("Known candidate recall:", report["known_candidate_recall"])
display(pd.read_csv(OUTPUT / "validation_slices.csv"))
errors = pd.read_csv(OUTPUT / "validation_predictions.csv")
errors["delta_ap5"] = errors.abhay_fusion_ap5 - errors.dalvin_rule_ap5
display(errors.sort_values("delta_ap5").head(15))
display(errors.sort_values("delta_ap5", ascending=False).head(15))
display(pd.read_csv(OUTPUT / "fusion_coefficients.csv"))


,map5,known_map5,unknown_map5,known_new_at1,unknown_new_at1
dalvin_rule,0.911397,0.889207,0.960722,0.111748,0.923567
detector_only,0.906456,0.890162,0.942675,0.108883,0.885350
abhay_fusion,0.908762,0.890640,0.949045,0.103152,0.898089


Fusion minus baseline MAP@5: -0.0026350461133068936
Queries: 506 Unknown fraction: 0.3102766798418972
Known candidate recall: 0.9799426934097422


,slice,n,dalvin_rule,detector_only,abhay_fusion
0,"{'species': 'hyena', 'unknown': np.False_}",158,0.903376,0.905485,0.903376
1,"{'species': 'hyena', 'unknown': np.True_}",79,0.940928,0.924051,0.936709
2,"{'species': 'leopard', 'unknown': np.False_}",191,0.877487,0.877487,0.880105
3,"{'species': 'leopard', 'unknown': np.True_}",78,0.980769,0.961538,0.961538
4,"{'viewpoint': 'back', 'unknown': np.False_}",21,0.932540,0.908730,0.932540
5,"{'viewpoint': 'back', 'unknown': np.True_}",10,0.950000,0.950000,0.950000
6,"{'viewpoint': 'backleft', 'unknown': np.False_}",6,0.916667,0.916667,0.916667
7,"{'viewpoint': 'backleft', 'unknown': np.True_}",1,1.000000,1.000000,1.000000
8,"{'viewpoint': 'backright', 'unknown': np.False_}",6,1.000000,1.000000,1.000000
9,"{'viewpoint': 'backright', 'unknown': np.True_}",3,0.666667,0.666667,0.666667


,id,identity,species,viewpoint,truth,unknown,candidate_hit,dalvin_rule_p_new,dalvin_rule_ap5,dalvin_rule_predictions,detector_only_p_new,detector_only_ap5,detector_only_predictions,abhay_fusion_p_new,abhay_fusion_ap5,abhay_fusion_predictions,delta_ap5
443,tr_01043,LEO0541,leopard,NaN,new_individual,True,False,0.610640,1.0,new_individual LEO0454 LEO0455 LEO0232 LEO0230,0.245791,0.5,LEO0454 new_individual LEO0455 LEO0232 LEO0230,0.243836,0.500000,LEO0454 new_individual LEO0455 LEO0232 LEO0230,-0.500000
212,tr_00467,HYE0261,hyena,right,new_individual,True,False,0.541058,1.0,new_individual HYE0034 HYE0432 HYE0041 HYE0384,0.248957,0.5,HYE0034 new_individual HYE0432 HYE0041 HYE0384,0.337830,0.500000,HYE0034 new_individual HYE0432 HYE0041 HYE0384,-0.500000
215,tr_05158,HYE0261,hyena,right,new_individual,True,False,0.584262,1.0,new_individual HYE0034 HYE0241 HYE0432 HYE0236,0.238445,0.5,HYE0034 new_individual HYE0241 HYE0432 HYE0236,0.410328,0.500000,HYE0034 new_individual HYE0241 HYE0432 HYE0236,-0.500000
13,tr_01973,HYE0446,hyena,right,HYE0446,False,True,0.473161,1.0,HYE0446 new_individual HYE0384 HYE0041 HYE0241,0.566041,0.5,new_individual HYE0446 HYE0384 HYE0041 HYE0241,0.644859,0.500000,new_individual HYE0446 HYE0384 HYE0041 HYE0241,-0.500000
495,tr_01519,LEO0462,leopard,left,new_individual,True,False,0.504698,1.0,new_individual LEO0605 LEO0200 LEO0455 LEO0466,0.385387,0.5,LEO0605 new_individual LEO0200 LEO0455 LEO0466,0.416234,0.500000,LEO0605 new_individual LEO0200 LEO0455 LEO0466,-0.500000
487,tr_02268,LEO0553,leopard,NaN,new_individual,True,False,0.514244,1.0,new_individual LEO0454 LEO0322 LEO0605 LEO0143,0.429503,0.5,LEO0454 new_individual LEO0322 LEO0605 LEO0143,0.330202,0.500000,LEO0454 new_individual LEO0322 LEO0605 LEO0143,-0.500000
130,tr_05453,HYE0236,hyena,left,HYE0236,False,True,0.000429,0.5,HYE0346 HYE0236 new_individual HYE0034 HYE0179,0.000858,0.5,HYE0346 HYE0236 new_individual HYE0034 HYE0179,0.060010,0.333333,HYE0346 new_individual HYE0236 HYE0034 HYE0179,-0.166667
504,tr_02870,LEO0562,leopard,right,new_individual,True,False,0.993362,1.0,new_individual LEO0502 LEO0116 LEO0143 LEO0669,0.989450,1.0,new_individual LEO0502 LEO0116 LEO0143 LEO0669,0.904714,1.000000,new_individual LEO0502 LEO0116 LEO0143 LEO0669,0.000000
15,tr_03787,HYE0446,hyena,right,HYE0446,False,True,0.841318,0.5,new_individual HYE0446 HYE0384 HYE0241 HYE0111,0.829894,0.5,new_individual HYE0446 HYE0384 HYE0241 HYE0111,0.914112,0.500000,new_individual HYE0446 HYE0384 HYE0241 HYE0111,0.000000
480,tr_05375,LEO0404,leopard,left,new_individual,True,False,0.998189,1.0,new_individual LEO0502 LEO0454 LEO0116 LEO0143,0.996727,1.0,new_individual LEO0502 LEO0454 LEO0116 LEO0143,0.928375,1.000000,new_individual LEO0502 LEO0454 LEO0116 LEO0143,0.000000


,id,identity,species,viewpoint,truth,unknown,candidate_hit,dalvin_rule_p_new,dalvin_rule_ap5,dalvin_rule_predictions,detector_only_p_new,detector_only_ap5,detector_only_predictions,abhay_fusion_p_new,abhay_fusion_ap5,abhay_fusion_predictions,delta_ap5
161,tr_03785,HYE0447,hyena,frontright,new_individual,True,False,0.277451,0.333333,HYE0066 HYE0301 new_individual HYE0034 HYE0115,0.554903,1.0,new_individual HYE0066 HYE0301 HYE0034 HYE0115,0.552399,1.0,new_individual HYE0066 HYE0301 HYE0034 HYE0115,0.666667
127,tr_02714,HYE0300,hyena,left,HYE0300,False,True,0.515580,0.500000,new_individual HYE0300 HYE0199 HYE0241 HYE0384,0.308633,1.0,HYE0300 new_individual HYE0199 HYE0241 HYE0384,0.317888,1.0,HYE0300 new_individual HYE0199 HYE0241 HYE0384,0.500000
398,tr_05706,LEO0230,leopard,NaN,LEO0230,False,True,0.562848,0.500000,new_individual LEO0230 LEO0143 LEO0322 LEO0502,0.245597,1.0,LEO0230 new_individual LEO0143 LEO0322 LEO0502,0.264698,1.0,LEO0230 new_individual LEO0143 LEO0322 LEO0502,0.500000
8,tr_05604,HYE0446,hyena,up,HYE0446,False,True,0.252636,0.333333,HYE0034 new_individual HYE0446 HYE0236 HYE0532,0.076414,0.5,HYE0034 HYE0446 new_individual HYE0236 HYE0532,0.137032,0.5,HYE0034 HYE0446 new_individual HYE0236 HYE0532,0.166667
480,tr_05375,LEO0404,leopard,left,new_individual,True,False,0.998189,1.000000,new_individual LEO0502 LEO0454 LEO0116 LEO0143,0.996727,1.0,new_individual LEO0502 LEO0454 LEO0116 LEO0143,0.928375,1.0,new_individual LEO0502 LEO0454 LEO0116 LEO0143,0.000000
490,tr_03093,LEO0646,leopard,right,new_individual,True,False,0.775367,1.000000,new_individual LEO0048 LEO0419 LEO0588 LEO0116,0.616833,1.0,new_individual LEO0048 LEO0419 LEO0588 LEO0116,0.696259,1.0,new_individual LEO0048 LEO0419 LEO0588 LEO0116,0.000000
0,tr_01705,HYE0446,hyena,backright,HYE0446,False,True,0.012705,1.000000,HYE0446 new_individual HYE0532 HYE0115 HYE0056,0.019059,1.0,HYE0446 new_individual HYE0532 HYE0115 HYE0056,0.055247,1.0,HYE0446 new_individual HYE0532 HYE0115 HYE0056,0.000000
481,tr_02709,LEO0404,leopard,left,new_individual,True,False,0.980377,1.000000,new_individual LEO0502 LEO0143 LEO0116 LEO0161,0.974709,1.0,new_individual LEO0502 LEO0143 LEO0116 LEO0161,0.905813,1.0,new_individual LEO0502 LEO0143 LEO0116 LEO0161,0.000000
482,tr_05608,LEO0404,leopard,left,new_individual,True,False,0.991635,1.000000,new_individual LEO0502 LEO0310 LEO0143 LEO0116,0.987067,1.0,new_individual LEO0502 LEO0310 LEO0143 LEO0116,0.915546,1.0,new_individual LEO0502 LEO0310 LEO0143 LEO0116,0.000000
483,tr_02836,LEO0404,leopard,left,new_individual,True,False,0.997876,1.000000,new_individual LEO0502 LEO0143 LEO0310 LEO0116,0.996816,1.0,new_individual LEO0502 LEO0143 LEO0310 LEO0116,0.943111,1.0,new_individual LEO0502 LEO0143 LEO0310 LEO0116,0.000000


,feature,standardized_coefficient
0,detector,1.327111
1,legacy_new,0.862491
2,raw_sum,0.125924
3,raw_max,-0.321756
4,pc_max,-0.225654
5,pc_gap,-0.060918
6,pc_entropy,-0.307075
7,raw_similarity,0.189777
8,lda_similarity,-0.399386
9,inliers,-0.083491


## 6. Archive results
The archive is saved locally beside this run. Keep it with your notebook/source version and record actual metrics in the contribution log. Test CSVs are generated only if PREDICT_TEST=True before the training cell. Their predictions can later be uploaded manually to Kaggle.


In [9]:
import shutil
archive = shutil.make_archive(str(RUN_DIR / "abhay_experiment"), "zip", OUTPUT)
print("Results archive:", archive)
print("Experiment documentation:", SCRIPT_DIR / "README.md")
print("Available submissions:", [p.name for p in OUTPUT.glob("submission_*.csv")])


Results archive: C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay\local_runs\20261124_193658_932524\abhay_experiment.zip
Experiment documentation: C:\Users\SANGEETA-DAILY\Documents\Abhay\Semester 5\ML\Project\Learning\Models\Abhay\local_runs\20261124_193658_932524\code\README.md
Available submissions: []
